# 02 - Document Ingestion Pipeline

This notebook demonstrates the LlamaIndex ingestion pipeline that:
1. Loads scanned PDF agreements (base contracts + amendments)
2. Extracts amendment-aware metadata from filenames (`contract_id`, `source_type`, `version`, `effective_date`)
3. **Structure-aware chunking** with `ContractNodeParser` — splits at section boundaries and enriches chunks with `section_type`, `has_table`, `clause_number`
4. Embeds and upserts to Pinecone

**Prerequisites:** Run `01_generate_data.ipynb` first to create the scanned PDFs.

In [1]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path
from dotenv import load_dotenv

# Load .env from project root (needed when running from notebooks/ directory)
load_dotenv(Path('../.env'))

from contract_lens.config import get_settings
from contract_lens.observability import init_observability

In [2]:
settings = get_settings()
init_observability(settings)
print(f'LLM deployment:       {settings.azure_openai_deployment}')
print(f'Embedding deployment: {settings.azure_openai_embedding_deployment}')
print(f'Pinecone index:       {settings.pinecone_index_name}')
print(f'LangFuse enabled:     {settings.langfuse_enabled}')

LLM deployment:       gpt-5.2
Embedding deployment: text-embedding-3-small
Pinecone index:       contract-lens
LangFuse enabled:     True


## Step 1: Load Scanned PDFs

`load_documents()` checks if Azure Document Intelligence is configured (endpoint + key in `.env`):
- **With Azure DI:** OCR processes scanned PDFs using the `prebuilt-layout` model, extracting text as markdown
- **Without Azure DI:** Falls back to `SimpleDirectoryReader` (works for text PDFs, but scanned image-PDFs will have no extractable text)

Each file becomes a `Document` object with metadata.

In [3]:
from contract_lens.ingestion.reader import load_documents

data_path = Path('../data/scans')
documents = load_documents(settings, data_path)

print(f'Loaded {len(documents)} documents from {data_path}')
print(f'\nFirst document metadata: {documents[0].metadata}')
print(f'First document text (first 300 chars):\n{documents[0].text[:300]}')

Loaded 9 documents from ../data/scans

First document metadata: {'file_name': '01_ITSVC-001_base_en_v1_2025-01-15.pdf'}
First document text (first 300 chars):
<!-- PageHeader="IT SERVICE AGREEMENT" -->


# IT SERVICE AGREEMENT

Contract ID: ITSVC-001

Effective Date: 2025-01-15

Version: 1 (Base Contract)

This IT Service Agreement (the 'Agreement') is entered into as of 2025-01-15 between TechFlow
Solutions Ltd., a company registered in London, United Ki


## Step 2: Enrich Metadata

We parse the filename convention to extract amendment-aware metadata:

```
{nn}_{contract_id}_{source_type}_{lang}_v{version}_{effective_date}.pdf
```

This gives us: `contract_id`, `source_type` (base/amendment), `language`, `version`, `effective_date`.

In [4]:
from contract_lens.ingestion.pipeline import parse_filename_metadata

for doc in documents:
    filename = doc.metadata.get('file_name', '')
    meta = parse_filename_metadata(filename)
    doc.metadata.update(meta)

# Show metadata summary
seen = set()
for doc in documents:
    fn = doc.metadata.get('file_name', '')
    if fn not in seen:
        seen.add(fn)
        print(f"{fn}")
        print(f"  contract_id={doc.metadata['contract_id']}  source_type={doc.metadata['source_type']}  "
              f"lang={doc.metadata['language']}  v={doc.metadata['version']}  date={doc.metadata['effective_date']}")


01_ITSVC-001_base_en_v1_2025-01-15.pdf
  contract_id=ITSVC-001  source_type=base  lang=en  v=1  date=2025-01-15
02_ITSVC-001_amendment_en_v2_2025-07-01.pdf
  contract_id=ITSVC-001  source_type=amendment  lang=en  v=2  date=2025-07-01
03_ITSVC-001_amendment_en_v3_2026-01-01.pdf
  contract_id=ITSVC-001  source_type=amendment  lang=en  v=3  date=2026-01-01
04_NDA-001_base_en_v1_2025-03-01.pdf
  contract_id=NDA-001  source_type=base  lang=en  v=1  date=2025-03-01
05_LEASE-001_base_pl_v1_2025-02-10.pdf
  contract_id=LEASE-001  source_type=base  lang=pl  v=1  date=2025-02-10
06_LEASE-001_amendment_pl_v2_2025-09-01.pdf
  contract_id=LEASE-001  source_type=amendment  lang=pl  v=2  date=2025-09-01
07_SLA-001_base_en_v1_2025-04-01.pdf
  contract_id=SLA-001  source_type=base  lang=en  v=1  date=2025-04-01
08_SLA-001_amendment_en_v2_2025-10-01.pdf
  contract_id=SLA-001  source_type=amendment  lang=en  v=2  date=2025-10-01
09_EMP-001_base_pl_v1_2025-05-01.pdf
  contract_id=EMP-001  source_type=base

## Step 3: Structure-Aware Chunking

We use `ContractNodeParser` instead of a generic splitter. It:
- **Detects section headings** (numbered sections, annexes, amendments, Polish legal markers like §, Rozdział)
- **Splits at section boundaries** to preserve clause context
- **Enriches metadata**: `section_type`, `section_name`, `has_table`, `clause_number`
- Falls back to `SentenceSplitter(1024/128)` for sections exceeding chunk size

In [5]:
from contract_lens.ingestion.node_parser import ContractNodeParser

parser = ContractNodeParser(chunk_size=1024, chunk_overlap=128)
nodes = parser.get_nodes_from_documents(documents)

print(f'{len(documents)} pages -> {len(nodes)} chunks')
print(f'\nSample chunk (node 0):')
print(f'  Text length:   {len(nodes[0].text)} chars')
print(f'  section_type:  {nodes[0].metadata.get("section_type", "")}')
print(f'  section_name:  {nodes[0].metadata.get("section_name", "")}')
print(f'  has_table:     {nodes[0].metadata.get("has_table", "")}')
print(f'  clause_number: {nodes[0].metadata.get("clause_number", "")}')
print(f'  Text:          {nodes[0].text[:200]}...')

9 pages -> 65 chunks

Sample chunk (node 0):
  Text length:   345 chars
  section_type:  general
  section_name:  IT SERVICE AGREEMENT
  has_table:     false
  clause_number: 
  Text:          IT SERVICE AGREEMENT
Contract ID: ITSVC-001

Effective Date: 2025-01-15

Version: 1 (Base Contract)

This IT Service Agreement (the 'Agreement') is entered into as of 2025-01-15 between TechFlow
Solut...


## Step 4: Embed and Upsert to Pinecone

Azure OpenAI `text-embedding-3-small` creates vector embeddings.
These are upserted to Pinecone along with the text and amendment-aware metadata.

In [6]:
from llama_index.core import StorageContext, VectorStoreIndex
from contract_lens.ingestion.pipeline import build_embedding_model, build_pinecone_vector_store

embed_model = build_embedding_model(settings)
vector_store = build_pinecone_vector_store(settings)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes,
    embed_model=embed_model,
    storage_context=storage_context,
    show_progress=True,
)

print(f'\nSuccessfully embedded and upserted {len(nodes)} chunks to Pinecone.')

Generating embeddings:   0%|          | 0/65 [00:00<?, ?it/s]

Upserted vectors:   0%|          | 0/65 [00:00<?, ?it/s]


Successfully embedded and upserted 65 chunks to Pinecone.


## Step 5: Quick Verification

Run a quick query to verify the index is working.

In [7]:
from contract_lens.retrieval.query_engine import query_contracts

answer = query_contracts(settings, question='What types of agreements are in the knowledge base?')
print(answer)

The knowledge base includes these types of agreements:

- Mutual Non-Disclosure Agreement (NDA)
- Service Level Agreement (SLA)
- IT Services Agreement (includes a confidentiality section)


## One-Shot Alternative

All the steps above are wrapped in a single function for production use:

In [8]:
# from contract_lens.ingestion.pipeline import run_ingestion
# num_nodes = run_ingestion(settings, data_dir='../data/scans')
# print(f'Ingested {num_nodes} chunks')

## Summary

| Step | Component | Result |
|------|-----------|--------|
| Load | `load_documents()` (Azure DI OCR or SimpleDirectoryReader) | PDF text as Documents |
| Metadata | `parse_filename_metadata` | `contract_id`, `source_type`, `language`, `version`, `effective_date` |
| Chunk | `ContractNodeParser(1024/128)` | Structure-aware nodes with `section_type`, `has_table`, `clause_number` |
| Embed + Store | `AzureOpenAIEmbedding` + `PineconeVectorStore` | Vectors in Pinecone with full metadata |

**Next:** Run `03_agent_demo.ipynb` to interact with the LangGraph agent, or `04_smart_chunking_demo.ipynb` to explore the structure-aware chunking in detail.